In [ ]:
import os

PASTA_DATASET = './Train_and_Validation'

# Classes S (selado) — índices 1,4,5,6,8,9,10,13,14,15,18 na ordem sorted do dataset
CLASSES_S = {
    '93000003', '93000009', '93000019', '93000020', '93000027',
    '93000030', '93000038', '93000088', '93000089', '93000095', '93000112',
}

extensoes = ('.jpg', '.jpeg', '.png', '.bmp')

todas   = sorted(d for d in os.listdir(PASTA_DATASET)
                 if os.path.isdir(os.path.join(PASTA_DATASET, d)))
classes = [d for d in todas if d.split('_')[0] in CLASSES_S]

print(f'{len(classes)} classes S carregadas:')
for i, c in enumerate(classes, 1):
    pasta = os.path.join(PASTA_DATASET, c)
    imgs  = [f for f in os.listdir(pasta)
             if f.lower().endswith(extensoes) and 'Zone' not in f]
    print(f'  {i:>2}. {c}  ({len(imgs)} imagens)')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

#IMG_ANALISE = './Train_and_Validation/93000003_Asas_Resfriado_Selado/TesteTempoPosicao2025-02-18 12_24_27.094767.jpg'
IMG_ANALISE = './Train_and_Validation\93000003_Asas_Resfriado_Selado\TesteTempoPosicao2025-02-18 12_25_22.849811.jpg'
img = cv2.imread(IMG_ANALISE)
assert img is not None, f'Erro ao carregar {IMG_ANALISE}'
h, w = img.shape[:2]
print(f'Dimensões: {w}x{h}')

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('93000003 — TesteTempoPosicao2025-02-18 12_24_27')
plt.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
# Diagnóstico da imagem — canais, histogramas, estatísticas
import os

# --- Metadados de arquivo ---
path = IMG_ANALISE
stat = os.stat(path)
print('=== METADADOS ===')
print(f'Arquivo  : {os.path.basename(path)}')
print(f'Tamanho  : {stat.st_size/1024:.1f} KB')
print(f'Dimensões: {img.shape[1]}x{img.shape[0]} px   canais={img.shape[2]}')
print(f'Dtype    : {img.dtype}')

# --- Estatísticas globais por canal BGR ---
print('\n=== ESTATÍSTICAS POR CANAL (BGR) ===')
nomes = ['Blue', 'Green', 'Red']
for i, nome in enumerate(nomes):
    c = img[:, :, i]
    print(f'  {nome}: min={c.min():3d}  max={c.max():3d}  mean={c.mean():.1f}  std={c.std():.1f}  '
          f'  p5={int(np.percentile(c,5)):3d}  p50={int(np.percentile(c,50)):3d}  p95={int(np.percentile(c,95)):3d}')

# --- Estatísticas em LAB e HSV ---
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab3 = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
print('\n=== HSV ===')
for i, nome in enumerate(['Hue','Sat','Val']):
    c = hsv[:,:,i]
    print(f'  {nome}: min={c.min():3d}  max={c.max():3d}  mean={c.mean():.1f}  std={c.std():.1f}  '
          f'  p5={int(np.percentile(c,5)):3d}  p50={int(np.percentile(c,50)):3d}  p95={int(np.percentile(c,95)):3d}')
print('\n=== LAB ===')
for i, nome in enumerate(['L','A','B']):
    c = lab3[:,:,i]
    print(f'  {nome}: min={c.min():3d}  max={c.max():3d}  mean={c.mean():.1f}  std={c.std():.1f}  '
          f'  p5={int(np.percentile(c,5)):3d}  p50={int(np.percentile(c,50)):3d}  p95={int(np.percentile(c,95)):3d}')

# --- Fração de pixels "quase branco" e "quase preto" ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
total = gray.size
print(f'\n=== DISTRIBUIÇÃO DE TONS (escala de cinza) ===')
for th_lo, th_hi, label in [(0,50,'muito escuro'), (50,100,'escuro'), (100,150,'meio-escuro'),
                              (150,200,'médio-claro'), (200,240,'claro'), (240,256,'quase branco')]:
    n = int(((gray >= th_lo) & (gray < th_hi)).sum())
    print(f'  [{th_lo:3d}-{th_hi:3d}] {label:<14}: {n:>7} px  ({100*n/total:.1f}%)')

# --- Visualização: todos os canais lado a lado ---
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

canais_bgr = cv2.split(img)
for i, (c, nome) in enumerate(zip(canais_bgr, ['Blue','Green','Red'])):
    axes[0][i].imshow(c, cmap='gray', vmin=0, vmax=255)
    axes[0][i].set_title(f'Canal {nome}'); axes[0][i].axis('off')

canais_hsv = cv2.split(hsv)
nomes_hsv  = ['Hue','Saturation','Value']
for i, (c, nome) in enumerate(zip(canais_hsv, nomes_hsv)):
    axes[1][i].imshow(c, cmap='gray' if i > 0 else 'hsv', vmin=0, vmax=255)
    axes[1][i].set_title(f'HSV — {nome}'); axes[1][i].axis('off')

canais_lab = cv2.split(lab3)
nomes_lab  = ['L (luminância)', 'A (verde↔vermelho)', 'B (azul↔amarelo)']
for i, (c, nome) in enumerate(zip(canais_lab, nomes_lab)):
    axes[2][i].imshow(c, cmap='gray', vmin=0, vmax=255)
    axes[2][i].set_title(f'LAB — {nome}'); axes[2][i].axis('off')

plt.suptitle('Diagnóstico de canais — BGR / HSV / LAB', fontsize=13)
plt.tight_layout(); plt.show()

# --- Histograma unificado (L, R, G, B, Sat) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for c, nome, cor in zip(cv2.split(img), ['Blue','Green','Red'], ['blue','green','red']):
    axes[0].plot(cv2.calcHist([c],[0],None,[256],[0,256]).flatten(), color=cor, alpha=0.7, label=nome)
axes[0].plot(cv2.calcHist([gray],[0],None,[256],[0,256]).flatten(), color='black', linewidth=2, label='Gray')
axes[0].set_title('Histograma BGR + Gray'); axes[0].legend(); axes[0].set_xlim(0,255)

axes[1].plot(cv2.calcHist([lab3],[0],None,[256],[0,256]).flatten(), color='black', label='L')
axes[1].plot(cv2.calcHist([hsv],[1],None,[256],[0,256]).flatten(),  color='orange',label='Sat')
axes[1].set_title('Histograma L (LAB) + Saturação (HSV)'); axes[1].legend(); axes[1].set_xlim(0,255)
plt.tight_layout(); plt.show()

In [ ]:
# Pré-processamento correto para imagem acromática (sem cor)
# Imagem 100% cinza: B=G=R, Sat=0 — abordagem puramente de luminância local
#
# Etapa 1: Estiramento linear [p5→p95] → [0→255]  (usa o range completo)
# Etapa 2: Top-hat pequeno (~15px)                  (captura picos locais = texto branco)
# Etapa 3: Limiarização adaptativa local             (binariza o texto contra o cinza)

P_LOW  = 5
P_HIGH = 95
TH_K   = 7
ADAPT_BLOCK = 21
ADAPT_C     = 50

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

p_lo  = int(np.percentile(gray, P_LOW))
p_hi  = int(np.percentile(gray, P_HIGH))
lut_stretch = np.clip(np.arange(256, dtype=np.float32) - p_lo, 0, None) / (p_hi - p_lo) * 255
lut_stretch = np.clip(lut_stretch, 0, 255).astype(np.uint8)
gray_stretch = cv2.LUT(gray, lut_stretch)
print(f'Estiramento: [{p_lo}, {p_hi}] → [0, 255]')
print(f'  antes:  mean={gray.mean():.1f}  std={gray.std():.1f}  p95={int(np.percentile(gray,95))}')
print(f'  depois: mean={gray_stretch.mean():.1f}  std={gray_stretch.std():.1f}  p95={int(np.percentile(gray_stretch,95))}')

k_th   = cv2.getStructuringElement(cv2.MORPH_RECT, (TH_K, TH_K))
tophat = cv2.morphologyEx(gray_stretch, cv2.MORPH_TOPHAT, k_th)
print(f'\nTop-hat k={TH_K}: max={tophat.max()}  mean={tophat.mean():.2f}  pixels>0: {int((tophat>0).sum())}')

thresh_adapt = cv2.adaptiveThreshold(
    tophat, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
    ADAPT_BLOCK, -ADAPT_C
)
px_texto = int((thresh_adapt > 0).sum())
print(f'Adaptive threshold: pixels texto={px_texto} ({100*px_texto/gray.size:.2f}%)')

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes[0][0].imshow(gray,         cmap='gray', vmin=0, vmax=255); axes[0][0].set_title('Gray original');     axes[0][0].axis('off')
axes[0][1].imshow(gray_stretch, cmap='gray', vmin=0, vmax=255); axes[0][1].set_title('Estirado [p5→p95]'); axes[0][1].axis('off')
axes[0][2].imshow(tophat,       cmap='hot',  vmin=0, vmax=255); axes[0][2].set_title(f'Top-hat k={TH_K}'); axes[0][2].axis('off')
axes[1][0].imshow(thresh_adapt, cmap='gray');                   axes[1][0].set_title('Adaptive threshold'); axes[1][0].axis('off')

overlay = cv2.cvtColor(gray_stretch, cv2.COLOR_GRAY2RGB)
overlay[thresh_adapt > 0] = [0, 220, 80]
axes[1][1].imshow(overlay);                                     axes[1][1].set_title('Overlay texto (verde)'); axes[1][1].axis('off')

axes[1][2].plot(cv2.calcHist([gray],         [0], None, [256], [0,256]).flatten(), color='gray',   label='Original')
axes[1][2].plot(cv2.calcHist([gray_stretch], [0], None, [256], [0,256]).flatten(), color='black',  label='Estirado')
axes[1][2].plot(cv2.calcHist([tophat],       [0], None, [256], [0,256]).flatten(), color='orange', label='Top-hat')
axes[1][2].set_title('Histogramas'); axes[1][2].legend(); axes[1][2].set_xlim(0,255)

plt.suptitle('Pré-processamento acromático — estiramento + top-hat + adaptive threshold', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Pipeline consolidado: ROI → máscara FPS → top-hat k=7 → limiares 40 e 50
# Parâmetros confirmados pelo usuário

ROI_ESQ   = 120
ROI_DIR   = 80
FPS_X0, FPS_X1 = 0, 180
FPS_Y0, FPS_Y1 = 0, 40

H, W = gray_stretch.shape

# 1. Recorte lateral
roi = gray_stretch[:, ROI_ESQ : W - ROI_DIR].copy()

# 2. Máscara FPS (coordenadas já na ROI — após corte de 120px esq)
fps_x0_roi = max(0, FPS_X0 - ROI_ESQ)
fps_x1_roi = max(0, FPS_X1 - ROI_ESQ)
roi[FPS_Y0:FPS_Y1, fps_x0_roi:fps_x1_roi] = 0

# 3. Top-hat k=7
k7       = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
th7_roi  = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k7)

print(f'Top-hat k=7 na ROI: max={th7_roi.max()}  mean={th7_roi.mean():.2f}')
for lim in [40, 50]:
    px = int((th7_roi > lim).sum())
    print(f'  limiar>{lim}: {px} px  ({100*px/roi.size:.2f}%)')

# 4. Visualização lado a lado: limiar 40 e 50
fig, axes = plt.subplots(2, 3, figsize=(22, 10))

axes[0][0].imshow(roi, cmap='gray', vmin=0, vmax=255)
axes[0][0].set_title('ROI + sem FPS (estirada)'); axes[0][0].axis('off')

axes[0][1].imshow(th7_roi, cmap='hot', vmin=0, vmax=100)
axes[0][1].set_title('Top-hat k=7'); axes[0][1].axis('off')

for col, lim in enumerate([40, 50], start=0):
    mask = th7_roi > lim
    ov   = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
    ov[mask] = [0, 220, 80]
    axes[1][col].imshow(ov)
    axes[1][col].set_title(f'Overlay limiar>{lim}  ({int(mask.sum())} px)')
    axes[1][col].axis('off')

# Diferença entre os dois limiares (pixels que estão em 40 mas não em 50)
diff_mask = (th7_roi > 40) & (th7_roi <= 50)
ov_diff   = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
ov_diff[th7_roi > 50]  = [0, 220, 80]    # verde  = ambos
ov_diff[diff_mask]     = [255, 180, 0]   # laranja = só no limiar 40
axes[0][2].imshow(ov_diff)
axes[0][2].set_title('Verde=ambos  Laranja=só em >40')
axes[0][2].axis('off')

axes[1][2].hist(th7_roi[th7_roi > 10].ravel(), 60, [10, 130], color='orange')
axes[1][2].axvline(40, color='green', linestyle='--', label='limiar 40')
axes[1][2].axvline(50, color='red',   linestyle='--', label='limiar 50')
axes[1][2].set_title('Histograma top-hat (valores > 10)')
axes[1][2].legend(); axes[1][2].set_xlim(10, 130)

plt.suptitle('Pipeline completo: ROI + FPS mask + top-hat k=7', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Componentes conectados — filtrar por área para isolar letras
# Traço ~7px, altura estimada ~25-40px → letra ≈ 150–600 px²
# Ruído pontual < 50 px²  |  blobs grandes (reflexos) > 3000 px²

AREA_MIN = 150
AREA_MAX = 600

# Máscara binária confirmada
bin_mask = (th7_roi > 50).astype(np.uint8) * 255

# Análise de componentes conectados
n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask, connectivity=8)
areas = stats[1:, cv2.CC_STAT_AREA]   # ignora background (label 0)

print(f'Total de componentes: {n_labels - 1}')
print(f'Distribuição de áreas:')
for lo, hi in [(1,10),(10,50),(50,150),(150,600),(600,3000),(3000,99999)]:
    n = int(((areas >= lo) & (areas < hi)).sum())
    print(f'  [{lo:5d} – {hi:5d}): {n:5d} componentes')

# Filtragem por área
mask_filtrada = np.zeros_like(bin_mask)
componentes_ok = []
for lbl in range(1, n_labels):
    area = stats[lbl, cv2.CC_STAT_AREA]
    if AREA_MIN <= area <= AREA_MAX:
        mask_filtrada[labels == lbl] = 255
        componentes_ok.append((lbl, area,
                               stats[lbl, cv2.CC_STAT_LEFT],
                               stats[lbl, cv2.CC_STAT_TOP],
                               stats[lbl, cv2.CC_STAT_WIDTH],
                               stats[lbl, cv2.CC_STAT_HEIGHT]))

print(f'\nApós filtro [{AREA_MIN}–{AREA_MAX}]: {len(componentes_ok)} componentes  '
      f'({int((mask_filtrada>0).sum())} px)')

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

axes[0].imshow(bin_mask, cmap='gray')
axes[0].set_title(f'Máscara bruta (>{50})  {int((bin_mask>0).sum())} px'); axes[0].axis('off')

axes[1].imshow(mask_filtrada, cmap='gray')
axes[1].set_title(f'Após filtro área [{AREA_MIN}–{AREA_MAX}]  {len(componentes_ok)} comp.'); axes[1].axis('off')

# Overlay com bboxes dos componentes sobre a ROI
ov = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
for lbl, area, x, y, w, h in componentes_ok:
    ov[labels == lbl] = [0, 220, 80]
    cv2.rectangle(ov, (x, y), (x+w, y+h), (255, 80, 0), 1)
axes[2].imshow(ov)
axes[2].set_title(f'Overlay componentes filtrados (verde=pixel  laranja=bbox)'); axes[2].axis('off')

plt.suptitle('Componentes conectados — filtro por área', fontsize=13)
plt.tight_layout(); plt.show()

# Histograma de áreas (escala log para ver toda a distribuição)
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(areas, bins=100, range=(150, 602), color='steelblue')
ax.axvline(AREA_MIN, color='green', linestyle='--', label=f'AREA_MIN={AREA_MIN}')
ax.axvline(AREA_MAX, color='red',   linestyle='--', label=f'AREA_MAX={AREA_MAX}')
ax.set_title('Distribuição de áreas dos componentes (0–3000 px²)')
ax.set_xlabel('Área (px²)'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Clustering por proximidade espacial — encontrar o grupo de componentes = texto
# Dois componentes sao vizinhos se a distancia entre centros for <= DIST_MAX

import math

DIST_MAX = 80   # px — ajustar se o cluster de texto nao fechar

comps = [(lbl, area, x, y, w, h, x + w // 2, y + h // 2)
         for lbl, area, x, y, w, h in componentes_ok]

visitado = [False] * len(comps)
clusters = []
for i in range(len(comps)):
    if visitado[i]:
        continue
    cluster = [i]
    visitado[i] = True
    fila = [i]
    while fila:
        cur = fila.pop()
        for j in range(len(comps)):
            if not visitado[j]:
                dx = comps[cur][6] - comps[j][6]
                dy = comps[cur][7] - comps[j][7]
                if math.sqrt(dx*dx + dy*dy) <= DIST_MAX:
                    visitado[j] = True
                    cluster.append(j)
                    fila.append(j)
    clusters.append(cluster)

clusters.sort(key=len, reverse=True)

print(f'DIST_MAX={DIST_MAX}px  ->  {len(clusters)} clusters')
for i, cl in enumerate(clusters):
    xs = [comps[j][2] for j in cl]; ys = [comps[j][3] for j in cl]
    ws = [comps[j][4] for j in cl]; hs = [comps[j][5] for j in cl]
    x0, y0 = min(xs), min(ys)
    x1 = max(x+w for x,w in zip(xs,ws))
    y1 = max(y+h for y,h in zip(ys,hs))
    areas_ = sorted([comps[j][1] for j in cl], reverse=True)
    print(f'  Cluster {i+1}: {len(cl):2d} comp  bbox=({x0:4d},{y0:3d},{x1-x0:3d}x{y1-y0:3d})  areas={areas_}')

cl_texto = clusters[0]
xs = [comps[j][2] for j in cl_texto]; ys = [comps[j][3] for j in cl_texto]
ws = [comps[j][4] for j in cl_texto]; hs = [comps[j][5] for j in cl_texto]
PAD = 10
tx  = max(0, min(xs) - PAD)
ty  = max(0, min(ys) - PAD)
tx2 = min(roi.shape[1], max(x+w for x,w in zip(xs,ws)) + PAD)
ty2 = min(roi.shape[0], max(y+h for y,h in zip(ys,hs)) + PAD)
print(f'\nTexto localizado (maior cluster): bbox ROI=({tx},{ty},{tx2-tx}x{ty2-ty})')

ov = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
cores_cl = [(255,80,80),(80,80,255),(255,165,0),(180,0,180),(0,180,180)]
for ci, cl in enumerate(clusters):
    cor = cores_cl[min(ci, len(cores_cl)-1)]
    for j in cl:
        _, area, x, y, w, h, cx, cy = comps[j]
        cv2.rectangle(ov, (x,y), (x+w,y+h), cor, 1)
        cv2.putText(ov, f'{j}({area})', (x, max(y-2,8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, cor, 1)

cv2.rectangle(ov, (tx, ty), (tx2, ty2), (0, 220, 80), 2)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
axes[0].imshow(ov)
axes[0].set_title(f'Clusters (verde=maior/texto  outras cores=ruido)  DIST_MAX={DIST_MAX}')
axes[0].axis('off')

recorte = roi[ty:ty2, tx:tx2]
axes[1].imshow(recorte, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Recorte do texto  ({tx2-tx}x{ty2-ty} px)')
axes[1].axis('off')

plt.suptitle('Localizacao do texto por clustering de proximidade', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# Lote — pipeline completo na classe classes[0] (93000003_Asas_Resfriado_Selado)
import math

# Parametros confirmados
P_LOW, P_HIGH    = 5, 95
ROI_ESQ, ROI_DIR = 120, 80
FPS_X0, FPS_X1   = 0, 180
FPS_Y0, FPS_Y1   = 0, 40
TH_K             = 7
LIMIAR           = 50
AREA_MIN         = 150
AREA_MAX         = 600
DIST_MAX         = 80

def pipeline(caminho):
    img = cv2.imread(caminho)
    if img is None:
        return None, None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    # Estiramento
    p_lo = int(np.percentile(gray, P_LOW))
    p_hi = int(np.percentile(gray, P_HIGH))
    if p_hi == p_lo:
        return None, None, img
    lut = np.clip((np.arange(256, dtype=np.float32) - p_lo) / (p_hi - p_lo) * 255, 0, 255).astype(np.uint8)
    gs  = cv2.LUT(gray, lut)

    # ROI + mascara FPS
    roi = gs[:, ROI_ESQ : W - ROI_DIR].copy()
    fps_x0 = max(0, FPS_X0 - ROI_ESQ)
    fps_x1 = max(0, FPS_X1 - ROI_ESQ)
    roi[FPS_Y0:FPS_Y1, fps_x0:fps_x1] = 0

    # Top-hat
    k7      = cv2.getStructuringElement(cv2.MORPH_RECT, (TH_K, TH_K))
    th7     = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k7)
    bin_mask = (th7 > LIMIAR).astype(np.uint8) * 255

    # Componentes conectados
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask, connectivity=8)
    comps_ok = []
    for lbl in range(1, n_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if AREA_MIN <= area <= AREA_MAX:
            x = stats[lbl, cv2.CC_STAT_LEFT]
            y = stats[lbl, cv2.CC_STAT_TOP]
            w = stats[lbl, cv2.CC_STAT_WIDTH]
            h = stats[lbl, cv2.CC_STAT_HEIGHT]
            comps_ok.append((lbl, area, x, y, w, h, x+w//2, y+h//2))

    if not comps_ok:
        return None, 0, img

    # Clustering
    visitado = [False] * len(comps_ok)
    clusters = []
    for i in range(len(comps_ok)):
        if visitado[i]:
            continue
        cl = [i]; visitado[i] = True; fila = [i]
        while fila:
            cur = fila.pop()
            for j in range(len(comps_ok)):
                if not visitado[j]:
                    dx = comps_ok[cur][6] - comps_ok[j][6]
                    dy = comps_ok[cur][7] - comps_ok[j][7]
                    if math.sqrt(dx*dx + dy*dy) <= DIST_MAX:
                        visitado[j] = True; cl.append(j); fila.append(j)
        clusters.append(cl)
    clusters.sort(key=len, reverse=True)

    cl_txt = clusters[0]
    xs = [comps_ok[j][2] for j in cl_txt]; ys = [comps_ok[j][3] for j in cl_txt]
    ws = [comps_ok[j][4] for j in cl_txt]; hs = [comps_ok[j][5] for j in cl_txt]
    PAD = 10
    tx  = max(0, min(xs) - PAD)
    ty  = max(0, min(ys) - PAD)
    tx2 = min(roi.shape[1], max(x+w for x,w in zip(xs,ws)) + PAD)
    ty2 = min(roi.shape[0], max(y+h for y,h in zip(ys,hs)) + PAD)
    bbox = (tx, ty, tx2-tx, ty2-ty)
    return bbox, len(cl_txt), img

# --- Lote ---
classe_dir = os.path.join(PASTA_DATASET, classes[0])
arquivos   = sorted(f for f in os.listdir(classe_dir)
                    if f.lower().endswith(extensoes) and 'Zone' not in f)

print(f'Classe: {classes[0]}')
print(f'Imagens: {len(arquivos)}\n')
print(f'  {"#":<3}  {"Arquivo":<52}  {"comp":>4}  {"bbox"}  {"status"}')
print('  ' + '-' * 90)

ok_count = falha_count = 0
for idx, nome in enumerate(arquivos, 1):
    caminho = os.path.join(classe_dir, nome)
    bbox, n_comp, _ = pipeline(caminho)
    if bbox:
        ok_count += 1
        status = 'OK'
        info   = f'({bbox[0]},{bbox[1]},{bbox[2]}x{bbox[3]})'
    else:
        falha_count += 1
        status = 'FALHA'
        info   = f'comp={n_comp}'
    print(f'  {idx:<3}  {nome[:52]:<52}  {(n_comp or 0):>4}  {info:<22}  {status}')

print(f'\nResumo: {ok_count} OK  |  {falha_count} FALHA  |  taxa={100*ok_count/len(arquivos):.1f}%')


In [ ]:
# Visualizacao: imagem original com bbox + recorte para cada imagem da classe

N_COLS   = 5   # imagens por linha
PAD_CROP = 10

resultados = []
for nome in arquivos:
    caminho = os.path.join(classe_dir, nome)
    bbox, n_comp, img_orig = pipeline(caminho)
    resultados.append((nome, bbox, n_comp, img_orig))

n = len(resultados)
n_rows = math.ceil(n / N_COLS)

# Grade 1: imagem ROI com bbox desenhada
fig1, axes1 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 4, n_rows * 3))
axes1 = axes1.flatten()

for idx, (nome, bbox, n_comp, img_orig) in enumerate(resultados):
    ax = axes1[idx]
    if img_orig is None:
        ax.set_facecolor('black'); ax.set_title(f'{idx+1} ERRO', fontsize=7); ax.axis('off'); continue

    H, W = img_orig.shape[:2]
    vis  = img_orig[:, ROI_ESQ : W - ROI_DIR].copy()
    vis  = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    if bbox:
        tx, ty, tw, th = bbox
        cv2.rectangle(vis, (tx, ty), (tx+tw, ty+th), (0, 220, 80), 2)
        status = 'OK'
    else:
        status = 'FALHA'
    ax.imshow(vis)
    ax.set_title(f'{idx+1} {status} comp={n_comp or 0}', fontsize=7)
    ax.axis('off')

for idx in range(n, len(axes1)):
    axes1[idx].axis('off')

plt.suptitle(f'ROI + bbox — {classes[0]}', fontsize=11)
plt.tight_layout(); plt.show()

# Grade 2: somente o recorte (crop) de cada imagem
fig2, axes2 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3, n_rows * 2.5))
axes2 = axes2.flatten()

for idx, (nome, bbox, n_comp, img_orig) in enumerate(resultados):
    ax = axes2[idx]
    if img_orig is None or bbox is None:
        ax.set_facecolor('#300'); ax.set_title(f'{idx+1} FALHA', fontsize=7, color='red')
        ax.axis('off'); continue

    H, W = img_orig.shape[:2]
    gray_orig = cv2.cvtColor(img_orig, cv2.COLOR_BGR2GRAY)
    p_lo = int(np.percentile(gray_orig, P_LOW))
    p_hi = int(np.percentile(gray_orig, P_HIGH))
    lut  = np.clip((np.arange(256, dtype=np.float32) - p_lo) / max(p_hi-p_lo,1) * 255, 0, 255).astype(np.uint8)
    gs   = cv2.LUT(gray_orig, lut)
    roi_ = gs[:, ROI_ESQ : W - ROI_DIR]

    tx, ty, tw, th = bbox
    crop = roi_[ty:ty+th, tx:tx+tw]
    ax.imshow(crop, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{idx+1}  {tw}x{th}', fontsize=7)
    ax.axis('off')

for idx in range(n, len(axes2)):
    axes2[idx].axis('off')

plt.suptitle(f'Recortes de texto — {classes[0]}', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Visualizacao horizontal — Grade 1: ROI+bbox  |  Grade 2: recorte

N_COLS = 5
n      = len(resultados_h)
n_rows = math.ceil(n / N_COLS)

# Grade 1 — ROI completa com bbox
fig1, axes1 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*4, n_rows*3))
axes1 = axes1.flatten()
for idx, (nome, bbox, n_comp, img_orig) in enumerate(resultados_h):
    ax = axes1[idx]
    if img_orig is None:
        ax.axis('off'); continue
    H, W = img_orig.shape[:2]
    vis = cv2.cvtColor(img_orig[:, ROI_ESQ:W-ROI_DIR], cv2.COLOR_BGR2RGB)
    if bbox:
        tx, ty, tw, th = bbox
        cv2.rectangle(vis, (tx,ty), (tx+tw,ty+th), (0,220,80), 2)
        status = 'OK'
    else:
        status = 'FALHA'
    ax.imshow(vis)
    ax.set_title(f'{idx+1} {status} comp={n_comp or 0}', fontsize=7)
    ax.axis('off')
for i in range(n, len(axes1)): axes1[i].axis('off')
plt.suptitle(f'ROI + bbox (horizontal) — {classes[0]}', fontsize=11)
plt.tight_layout(); plt.show()

# Grade 2 — somente o recorte do texto
fig2, axes2 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*3, n_rows*2.5))
axes2 = axes2.flatten()
for idx, (nome, bbox, n_comp, img_orig) in enumerate(resultados_h):
    ax = axes2[idx]
    if img_orig is None or bbox is None:
        ax.set_facecolor('#300')
        ax.set_title(f'{idx+1} FALHA', fontsize=7, color='red')
        ax.axis('off'); continue
    H, W = img_orig.shape[:2]
    gray_o = cv2.cvtColor(img_orig, cv2.COLOR_BGR2GRAY)
    p_lo = int(np.percentile(gray_o, P_LOW))
    p_hi = int(np.percentile(gray_o, P_HIGH))
    lut  = np.clip((np.arange(256, dtype=np.float32)-p_lo)/max(p_hi-p_lo,1)*255, 0, 255).astype(np.uint8)
    roi_ = cv2.LUT(gray_o, lut)[:, ROI_ESQ:W-ROI_DIR]
    tx, ty, tw, th = bbox
    ax.imshow(roi_[ty:ty+th, tx:tx+tw], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{idx+1}  {tw}x{th}', fontsize=7)
    ax.axis('off')
for i in range(n, len(axes2)): axes2[i].axis('off')
plt.suptitle(f'Recortes de texto (horizontal) — {classes[0]}', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Lote horizontal — pipeline com filtro proporcao w/h >= PROP_MIN_TEXT
# Seleciona o maior cluster cujo bbox seja mais largo que alto (texto = horizontal)
import math

P_LOW, P_HIGH    = 5, 95
ROI_ESQ, ROI_DIR = 120, 80
FPS_X0, FPS_X1   = 0, 180
FPS_Y0, FPS_Y1   = 0, 40
TH_K             = 7
LIMIAR           = 50
AREA_MIN         = 150

AREA_MAX         = 600
DIST_MAX         = 80
PROP_MIN_TEXT    = 1.0   # w/h minimo para considerar cluster de texto

def pipeline_h(caminho):
    img = cv2.imread(caminho)
    if img is None:
        return None, None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    p_lo = int(np.percentile(gray, P_LOW))
    p_hi = int(np.percentile(gray, P_HIGH))
    if p_hi == p_lo:
        return None, None, img
    lut = np.clip((np.arange(256, dtype=np.float32) - p_lo) / (p_hi - p_lo) * 255, 0, 255).astype(np.uint8)
    gs  = cv2.LUT(gray, lut)

    roi = gs[:, ROI_ESQ : W - ROI_DIR].copy()
    fps_x0 = max(0, FPS_X0 - ROI_ESQ)
    fps_x1 = max(0, FPS_X1 - ROI_ESQ)
    roi[FPS_Y0:FPS_Y1, fps_x0:fps_x1] = 0

    k7       = cv2.getStructuringElement(cv2.MORPH_RECT, (TH_K, TH_K))
    th7      = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k7)
    bin_mask = (th7 > LIMIAR).astype(np.uint8) * 255

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask, connectivity=8)
    comps_ok = []
    for lbl in range(1, n_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if AREA_MIN <= area <= AREA_MAX:
            x = stats[lbl, cv2.CC_STAT_LEFT]
            y = stats[lbl, cv2.CC_STAT_TOP]
            w = stats[lbl, cv2.CC_STAT_WIDTH]
            h = stats[lbl, cv2.CC_STAT_HEIGHT]
            comps_ok.append((lbl, area, x, y, w, h, x+w//2, y+h//2))

    if not comps_ok:
        return None, 0, img

    # Clustering
    visitado = [False] * len(comps_ok)
    clusters = []
    for i in range(len(comps_ok)):
        if visitado[i]:
            continue
        cl = [i]; visitado[i] = True; fila = [i]
        while fila:
            cur = fila.pop()
            for j in range(len(comps_ok)):
                if not visitado[j]:
                    dx = comps_ok[cur][6] - comps_ok[j][6]
                    dy = comps_ok[cur][7] - comps_ok[j][7]
                    if math.sqrt(dx*dx + dy*dy) <= DIST_MAX:
                        visitado[j] = True; cl.append(j); fila.append(j)
        clusters.append(cl)

    # Calcula bbox de cada cluster e filtra por proporcao horizontal
    def cluster_bbox(cl):
        xs = [comps_ok[j][2] for j in cl]; ys = [comps_ok[j][3] for j in cl]
        ws = [comps_ok[j][4] for j in cl]; hs = [comps_ok[j][5] for j in cl]
        x0, y0 = min(xs), min(ys)
        x1 = max(x+w for x,w in zip(xs,ws))
        y1 = max(y+h for y,h in zip(ys,hs))
        return x0, y0, x1-x0, y1-y0

    candidatos = []
    for cl in clusters:
        bx, by, bw, bh = cluster_bbox(cl)
        prop = bw / max(bh, 1)
        if prop >= PROP_MIN_TEXT:
            candidatos.append((cl, bw, bh, prop))

    if not candidatos:
        # fallback: maior cluster sem filtro
        clusters.sort(key=len, reverse=True)
        cl_txt = clusters[0]
    else:
        # maior cluster horizontal (por numero de componentes)
        candidatos.sort(key=lambda t: len(t[0]), reverse=True)
        cl_txt = candidatos[0][0]

    bx, by, bw, bh = cluster_bbox(cl_txt)
    PAD = 10
    tx  = max(0, bx - PAD)
    ty  = max(0, by - PAD)
    tx2 = min(roi.shape[1], bx + bw + PAD)
    ty2 = min(roi.shape[0], by + bh + PAD)
    return (tx, ty, tx2-tx, ty2-ty), len(cl_txt), img

# --- Lote ---
classe_dir = os.path.join(PASTA_DATASET, classes[0])
arquivos   = sorted(f for f in os.listdir(classe_dir)
                    if f.lower().endswith(extensoes) and 'Zone' not in f)

print(f'Classe: {classes[0]}')
print(f'  {"#":<3}  {"Arquivo":<52}  {"comp":>4}  {"bbox"}  {"status"}')
print('  ' + '-'*90)

resultados_h = []
ok_count = falha_count = 0
for idx, nome in enumerate(arquivos, 1):
    caminho = os.path.join(classe_dir, nome)
    bbox, n_comp, img_orig = pipeline_h(caminho)
    resultados_h.append((nome, bbox, n_comp, img_orig))
    if bbox:
        ok_count += 1
        bx,by,bw,bh = bbox
        prop = f'{bw/max(bh,1):.2f}'
        print(f'  {idx:<3}  {nome[:52]:<52}  {n_comp:>4}  ({bx},{by},{bw}x{bh})  OK  w/h={prop}')
    else:
        falha_count += 1
        print(f'  {idx:<3}  {nome[:52]:<52}  {(n_comp or 0):>4}  —  FALHA')

print(f'\nResumo: {ok_count} OK  |  {falha_count} FALHA  |  taxa={100*ok_count/len(arquivos):.1f}%')


In [ ]:
# Lote completo — todas as classes CLASSES_S com pipeline horizontal
import math

resultados_dataset = {}   # classe -> lista de (nome, bbox, n_comp, img_orig)

total_ok = total_falha = total_imgs = 0

for classe in classes:
    classe_dir = os.path.join(PASTA_DATASET, classe)
    arquivos_c = sorted(f for f in os.listdir(classe_dir)
                        if f.lower().endswith(extensoes) and 'Zone' not in f)

    res_classe = []
    ok_c = falha_c = 0
    for nome in arquivos_c:
        caminho = os.path.join(classe_dir, nome)
        bbox, n_comp, img_orig = pipeline_h(caminho)
        res_classe.append((nome, bbox, n_comp, img_orig))
        if bbox:
            ok_c += 1
        else:
            falha_c += 1

    resultados_dataset[classe] = res_classe
    total_ok    += ok_c
    total_falha += falha_c
    total_imgs  += len(arquivos_c)

    taxa = 100 * ok_c / max(len(arquivos_c), 1)
    flag = '' if falha_c == 0 else f'  *** {falha_c} FALHA(S) ***'
    print(f'{classe:<55}  {ok_c:>3}/{len(arquivos_c):>3}  ({taxa:5.1f}%){flag}')

print()
print(f'TOTAL: {total_ok} OK  |  {total_falha} FALHA  |  {total_imgs} imagens'
      f'  |  taxa geral={100*total_ok/max(total_imgs,1):.1f}%')


In [ ]:
# Salva visualizacoes em resultado_selado/ e exibe no notebook — limpa a pasta a cada execucao
import shutil

PASTA_SAIDA = './resultado_selado'
if os.path.exists(PASTA_SAIDA):
    shutil.rmtree(PASTA_SAIDA)
os.makedirs(PASTA_SAIDA)
print(f'Pasta limpa: {PASTA_SAIDA}\n')

N_COLS = 5

def square_crop(roi_, bbox):
    tx, ty, tw, th = bbox
    lado = max(tw, th)
    cx   = tx + tw // 2
    cy   = ty + th // 2
    x0   = max(0, cx - lado // 2)
    y0   = max(0, cy - lado // 2)
    x1   = min(roi_.shape[1], x0 + lado)
    y1   = min(roi_.shape[0], y0 + lado)
    return roi_[y0:y1, x0:x1]

for classe, res in resultados_dataset.items():
    n      = len(res)
    n_rows = math.ceil(n / N_COLS)
    nome_base = classe[:30]

    # Grade 1 — ROI com bbox
    fig1, axes1 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*4, n_rows*3))
    axes1 = axes1.flatten()
    for idx, (nome, bbox, n_comp, img_orig) in enumerate(res):
        ax = axes1[idx]
        if img_orig is None:
            ax.axis('off'); continue
        H, W = img_orig.shape[:2]
        vis = cv2.cvtColor(img_orig[:, ROI_ESQ:W-ROI_DIR], cv2.COLOR_BGR2RGB)
        if bbox:
            tx, ty, tw, th = bbox
            cv2.rectangle(vis, (tx,ty), (tx+tw,ty+th), (0,220,80), 2)
            status = 'OK'
        else:
            status = 'FALHA'
        ax.imshow(vis)
        ax.set_title(f'{idx+1} {status}', fontsize=7)
        ax.axis('off')
    for i in range(n, len(axes1)): axes1[i].axis('off')
    plt.suptitle(f'ROI + bbox — {classe}', fontsize=10)
    plt.tight_layout()
    fig1.savefig(os.path.join(PASTA_SAIDA, f'{nome_base}_1_roi_bbox.png'), dpi=80, bbox_inches='tight')
    plt.show()
    plt.close(fig1)

    # Grade 2 — crop quadrado
    fig2, axes2 = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*3, n_rows*3))
    axes2 = axes2.flatten()
    for idx, (nome, bbox, n_comp, img_orig) in enumerate(res):
        ax = axes2[idx]
        if img_orig is None or bbox is None:
            ax.set_facecolor('#300')
            ax.set_title(f'{idx+1} FALHA', fontsize=7, color='red')
            ax.axis('off'); continue
        H, W = img_orig.shape[:2]
        gray_o = cv2.cvtColor(img_orig, cv2.COLOR_BGR2GRAY)
        p_lo = int(np.percentile(gray_o, P_LOW))
        p_hi = int(np.percentile(gray_o, P_HIGH))
        lut  = np.clip((np.arange(256, dtype=np.float32)-p_lo)/max(p_hi-p_lo,1)*255, 0, 255).astype(np.uint8)
        roi_ = cv2.LUT(gray_o, lut)[:, ROI_ESQ:W-ROI_DIR]
        crop = square_crop(roi_, bbox)
        ax.imshow(crop, cmap='gray', vmin=0, vmax=255)
        tx, ty, tw, th = bbox
        ax.set_title(f'{idx+1}  sq{max(tw,th)}', fontsize=7)
        ax.axis('off')
    for i in range(n, len(axes2)): axes2[i].axis('off')
    plt.suptitle(f'Crops quadrados — {classe}', fontsize=10)
    plt.tight_layout()
    fig2.savefig(os.path.join(PASTA_SAIDA, f'{nome_base}_2_crops.png'), dpi=80, bbox_inches='tight')
    plt.show()
    plt.close(fig2)

    print(f'  {classe}  ->  salvo')

print(f'\nPronto. {len(resultados_dataset)*2} arquivos em {PASTA_SAIDA}/')


In [ ]:
# Dataset de teste pos-processamento: classe 0 (dificil) + classe 1 (facil)
# classes[0] = 93000003_Asas_Resfriado_Selado
# classes[1] = 93000009_Coxinhas_das_Asas_Congelado_Selado

CLASSES_TESTE = [classes[0], classes[1]]
N_COLS = 5

dataset_teste = {c: resultados_dataset[c] for c in CLASSES_TESTE}

for classe, res in dataset_teste.items():
    n      = len(res)
    n_rows = math.ceil(n / N_COLS)

    print(f'\n=== {classe} ===')
    print(f'  {"#":<3}  {"bbox (tx,ty,w,h)":<22}  {"w/h":>5}  {"sq lado":>7}')
    print('  ' + '-'*45)
    for idx, (nome, bbox, n_comp, _) in enumerate(res):
        if bbox:
            tx, ty, tw, th = bbox
            prop = tw / max(th, 1)
            lado = max(tw, th)
            print(f'  {idx+1:<3}  ({tx:4d},{ty:4d},{tw:3d}x{th:3d})      {prop:5.2f}  {lado:>7}')
        else:
            print(f'  {idx+1:<3}  FALHA')

    # Grade visual: ROI + bbox
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*4, n_rows*3))
    axes = axes.flatten()
    for idx, (nome, bbox, n_comp, img_orig) in enumerate(res):
        ax = axes[idx]
        if img_orig is None:
            ax.axis('off'); continue
        H, W = img_orig.shape[:2]
        vis = cv2.cvtColor(img_orig[:, ROI_ESQ:W-ROI_DIR], cv2.COLOR_BGR2RGB)
        if bbox:
            tx, ty, tw, th = bbox
            cv2.rectangle(vis, (tx,ty), (tx+tw,ty+th), (0,220,80), 2)
            status = f'OK {tw}x{th}'
        else:
            status = 'FALHA'
        ax.imshow(vis)
        ax.set_title(f'{idx+1} {status}', fontsize=7)
        ax.axis('off')
    for i in range(n, len(axes)): axes[i].axis('off')
    plt.suptitle(f'ROI + bbox — {classe}', fontsize=10)
    plt.tight_layout(); plt.show()


In [ ]:
# Pipeline debug — retorna componentes filtrados alem do bbox
# Usado para inspecionar o pre-box nas classes de teste

def pipeline_h_debug(caminho):
    img = cv2.imread(caminho)
    if img is None:
        return None, None, None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    p_lo = int(np.percentile(gray, P_LOW))
    p_hi = int(np.percentile(gray, P_HIGH))
    if p_hi == p_lo:
        return None, [], img, None
    lut = np.clip((np.arange(256, dtype=np.float32)-p_lo)/(p_hi-p_lo)*255, 0, 255).astype(np.uint8)
    gs  = cv2.LUT(gray, lut)

    roi = gs[:, ROI_ESQ:W-ROI_DIR].copy()
    fps_x0 = max(0, FPS_X0 - ROI_ESQ)
    fps_x1 = max(0, FPS_X1 - ROI_ESQ)
    roi[FPS_Y0:FPS_Y1, fps_x0:fps_x1] = 0

    k7       = cv2.getStructuringElement(cv2.MORPH_RECT, (TH_K, TH_K))
    th7      = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k7)
    bin_mask = (th7 > LIMIAR).astype(np.uint8) * 255

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask, connectivity=8)
    comps_ok = []
    for lbl in range(1, n_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if AREA_MIN <= area <= AREA_MAX:
            x = stats[lbl, cv2.CC_STAT_LEFT]
            y = stats[lbl, cv2.CC_STAT_TOP]
            w = stats[lbl, cv2.CC_STAT_WIDTH]
            h = stats[lbl, cv2.CC_STAT_HEIGHT]
            comps_ok.append((area, x, y, w, h))

    # reutiliza pipeline_h para o bbox final
    bbox, _, _ = pipeline_h(caminho)
    return bbox, comps_ok, img, roi

# Visualizacao: componentes (pre-box) + bbox final
N_COLS = 5

for classe in CLASSES_TESTE:
    classe_dir = os.path.join(PASTA_DATASET, classe)
    arquivos_c = sorted(f for f in os.listdir(classe_dir)
                        if f.lower().endswith(extensoes) and 'Zone' not in f)

    n      = len(arquivos_c)
    n_rows = math.ceil(n / N_COLS)

    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS*4, n_rows*3))
    axes = axes.flatten()

    for idx, nome in enumerate(arquivos_c):
        ax = axes[idx]
        caminho = os.path.join(classe_dir, nome)
        bbox, comps_ok, img_orig, roi_ = pipeline_h_debug(caminho)

        if roi_ is None:
            ax.axis('off'); continue

        vis = cv2.cvtColor(roi_, cv2.COLOR_GRAY2RGB)

        # componentes filtrados (laranja)
        for area, x, y, w, h in comps_ok:
            cv2.rectangle(vis, (x, y), (x+w, y+h), (255, 140, 0), 1)

        # bbox final (verde)
        if bbox:
            tx, ty, tw, th = bbox
            cv2.rectangle(vis, (tx,ty), (tx+tw,ty+th), (0,220,80), 2)

        n_comp = len(comps_ok)
        ax.imshow(vis)
        ax.set_title(f'{idx+1}  comp={n_comp}', fontsize=7)
        ax.axis('off')

    for i in range(n, len(axes)): axes[i].axis('off')
    plt.suptitle(f'Componentes pre-box (laranja) + bbox final (verde) — {classe}', fontsize=10)
    plt.tight_layout(); plt.show()
